# 🏠 Smart House Price Prediction
## Notebook 02 – Data Preprocessing

---

### 🎯 Objective

This notebook performs comprehensive data preprocessing on the Ames Housing dataset. The preprocessing pipeline is designed to prepare clean, consistent, and machine-learning-ready data for feature engineering and model development.

The preprocessing steps include:

- Missing value treatment
- Ordinal feature encoding
- Nominal feature encoding
- Feature scaling
- Data validation
- Saving processed datasets for downstream tasks

---

### 📌 Expected Outputs

At the end of this notebook, the following artifacts will be generated:

- Clean training dataset
- Encoded categorical features
- Scaled numerical features
- Preprocessing transformers (if applicable)
- Processed dataset ready for feature engineering and model training

---

### 🛠 Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- Joblib

---

### Project Structure

Smart-House-Price-Prediction/

├── backend/
│   └── utils/
│       └── preprocessing_pipeline.py
│
├── data/
│   ├── raw/
│   └── processed/
│
├── notebooks/
│   ├── 01_data_understanding.ipynb
│   └── 02_data_preprocessing.ipynb
│
└── models/

In [42]:
# ============================================================
# Notebook 02 : Data Preprocessing
# Import Required Libraries
# ============================================================

import os
import sys
import joblib
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")


Libraries imported successfully.


## ⚙️ Project Configuration

This section configures the notebook environment.

The objectives are:

- Detect the project root automatically.
- Configure Python's import path.
- Define dataset locations.
- Ensure reproducibility across different machines.
- Validate that required project folders exist before proceeding.

In [43]:
# ============================================================
# Project Configuration
# ============================================================

# Locate the project root directory
PROJECT_ROOT = Path.cwd().resolve().parent

print(f"Project Root : {PROJECT_ROOT}")

# ------------------------------------------------------------
# Define important project directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

BACKEND_DIR = PROJECT_ROOT / "backend"
UTILS_DIR = BACKEND_DIR / "utils"

MODELS_DIR = PROJECT_ROOT / "models"

# ------------------------------------------------------------
# Verify directories
# ------------------------------------------------------------

required_dirs = {
    "Raw Data": RAW_DATA_DIR,
    "Processed Data": PROCESSED_DATA_DIR,
    "Backend": BACKEND_DIR,
    "Utils": UTILS_DIR,
    "Models": MODELS_DIR
}

print("\nChecking project structure...\n")

for name, folder in required_dirs.items():
    if folder.exists():
        print(f"✓ {name:<15}: Found")
    else:
        print(f"✗ {name:<15}: Missing")


Project Root : C:\Users\HP\AI Projects\Smart-House-Price-Prediction

Checking project structure...

✓ Raw Data       : Found
✓ Processed Data : Found
✓ Backend        : Found
✓ Utils          : Found
✓ Models         : Found


## 📦 Configure Python Import Path

This section adds the project root to Python's module search path.

This allows us to import custom utility modules (such as the preprocessing pipeline)
from anywhere inside the project without modifying relative paths.

In [44]:
# ============================================================
# Configure Python Import Path
# ============================================================

project_root = str(PROJECT_ROOT)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("✓ Project root added to Python path.")


✓ Project root added to Python path.


## 🔧 Import Preprocessing Pipeline

Instead of writing preprocessing logic directly in this notebook, we import reusable
functions from the project's preprocessing module.

This keeps the notebook clean, modular, and consistent across training and inference.

In [45]:
# ============================================================
# Import Custom Preprocessing Functions
# ============================================================

try:
    from backend.utils.preprocessing_pipeline import (
        handle_missing_values,
        apply_ordinal_encoding,
        apply_nominal_encoding,
        scale_numerical_features,
    )

    print("✓ Custom preprocessing pipeline imported successfully.")

except Exception as e:
    print("✗ Failed to import preprocessing pipeline.")
    raise e


✓ Custom preprocessing pipeline imported successfully.


## 📂 Load Training Dataset

This section loads the Ames Housing training dataset from the project's raw data directory.

Before any preprocessing begins, we verify that:

- The dataset file exists.
- The dataset loads successfully.
- The expected target column (`SalePrice`) is present.
- The dataset dimensions are correct.
- The dataset is ready for preprocessing.

In [46]:
# ============================================================
# Load Training Dataset
# ============================================================

TRAIN_DATA_PATH = RAW_DATA_DIR / "train.csv"

if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Training dataset not found:\n{TRAIN_DATA_PATH}"
    )

df = pd.read_csv(TRAIN_DATA_PATH)

print("✓ Training dataset loaded successfully.\n")

print(f"Dataset Path   : {TRAIN_DATA_PATH}")
print(f"Rows           : {df.shape[0]}")
print(f"Columns        : {df.shape[1]}")
print(f"Memory Usage   : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


✓ Training dataset loaded successfully.

Dataset Path   : C:\Users\HP\AI Projects\Smart-House-Price-Prediction\data\raw\train.csv
Rows           : 1460
Columns        : 81
Memory Usage   : 3.43 MB


## ✅ Initial Dataset Validation

Before applying preprocessing, the dataset is validated to ensure structural integrity.

The following checks are performed:

- Dataset dimensions
- Presence of the target variable
- Duplicate records
- Missing values
- Data types
- Infinite values
- Object columns
- Numerical columns

In [47]:
# ============================================================
# Initial Dataset Validation
# ============================================================

print("=" * 70)
print("DATASET VALIDATION REPORT")
print("=" * 70)

# ------------------------------------------------------------
# Shape
# ------------------------------------------------------------

print(f"\nDataset Shape : {df.shape}")

# ------------------------------------------------------------
# Target Variable
# ------------------------------------------------------------

TARGET = "SalePrice"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

print(f"✓ Target Column : {TARGET}")

# ------------------------------------------------------------
# Duplicate Rows
# ------------------------------------------------------------

duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

# ------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------

missing = df.isnull().sum().sum()

print(f"Total Missing Values : {missing}")

# ------------------------------------------------------------
# Infinite Values
# ------------------------------------------------------------

numeric_df = df.select_dtypes(include=np.number)

infinite_values = np.isinf(numeric_df.to_numpy()).sum()

print(f"Infinite Values : {infinite_values}")

# ------------------------------------------------------------
# Column Types
# ------------------------------------------------------------

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

print(f"\nNumerical Features  : {len(numerical_cols)}")
print(f"Categorical Features: {len(categorical_cols)}")

print("\nValidation Status : PASSED ✅")


DATASET VALIDATION REPORT

Dataset Shape : (1460, 81)
✓ Target Column : SalePrice
Duplicate Rows : 0
Total Missing Values : 7829
Infinite Values : 0

Numerical Features  : 38
Categorical Features: 43

Validation Status : PASSED ✅


## 👀 Dataset Preview

To better understand the structure of the dataset, we inspect:

- First five records
- Last five records
- Random sample
- DataFrame summary

In [48]:
# ============================================================
# Dataset Preview
# ============================================================

print("First Five Records")
display(df.head())

print("\nLast Five Records")
display(df.tail())

print("\nRandom Sample")
display(df.sample(5, random_state=42))

print("\nDataset Information\n")
df.info()


First Five Records


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.00,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.00,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.00,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.00,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.00,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.00,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.00,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.00,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.00,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.00,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



Last Five Records


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1455,1456,60,RL,62.00,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,5,1999,2000,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,PConc,Gd,TA,No,Unf,0,Unf,0,953,953,GasA,Ex,Y,SBrkr,953,694,0,1647,0,0,2,1,3,1,TA,7,Typ,1,TA,Attchd,1999.00,RFn,2,460,TA,TA,Y,0,40,0,0,0,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.00,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,6,6,1978,1988,Gable,CompShg,Plywood,Plywood,Stone,119.00,TA,TA,CBlock,Gd,TA,No,ALQ,790,Rec,163,589,1542,GasA,TA,Y,SBrkr,2073,0,0,2073,1,0,2,0,3,1,TA,7,Min1,2,TA,Attchd,1978.00,Unf,2,500,TA,TA,Y,349,0,0,0,0,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.00,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,9,1941,2006,Gable,CompShg,CemntBd,CmentBd,NaN,0.00,Ex,Gd,Stone,TA,Gd,No,GLQ,275,Unf,0,877,1152,GasA,Ex,Y,SBrkr,1188,1152,0,2340,0,0,2,0,4,1,Gd,9,Typ,2,Gd,Attchd,1941.00,RFn,1,252,TA,TA,Y,0,60,0,0,0,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.00,9717,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1950,1996,Hip,CompShg,MetalSd,MetalSd,NaN,0.00,TA,TA,CBlock,TA,TA,Mn,GLQ,49,Rec,1029,0,1078,GasA,Gd,Y,FuseA,1078,0,0,1078,1,0,1,0,2,1,Gd,5,Typ,0,NaN,Attchd,1950.00,Unf,1,240,TA,TA,Y,366,0,112,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125
1459,1460,20,RL,75.00,9937,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Norm,Norm,1Fam,1Story,5,6,1965,1965,Gable,CompShg,HdBoard,HdBoard,NaN,0.00,Gd,TA,CBlock,TA,TA,No,BLQ,830,LwQ,290,136,1256,GasA,Gd,Y,SBrkr,1256,0,0,1256,1,0,1,1,3,1,TA,6,Typ,0,NaN,Attchd,1965.00,Fin,1,276,TA,TA,Y,736,68,0,0,0,0,NaN,NaN,NaN,0,6,2008,WD,Normal,147500



Random Sample


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
892,893,20,RL,70.00,8414,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,Norm,1Fam,1Story,6,8,1963,2003,Hip,CompShg,HdBoard,HdBoard,NaN,0.00,TA,TA,CBlock,TA,TA,No,GLQ,663,Unf,0,396,1059,GasA,TA,Y,SBrkr,1068,0,0,1068,0,1,1,0,3,1,TA,6,Typ,0,NaN,Attchd,1963.00,RFn,1,264,TA,TA,Y,192,0,0,0,0,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal,154500
1105,1106,60,RL,98.00,12256,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,1994,1995,Gable,CompShg,HdBoard,HdBoard,BrkFace,362.00,Gd,TA,PConc,Ex,TA,Av,GLQ,1032,Unf,0,431,1463,GasA,Ex,Y,SBrkr,1500,1122,0,2622,1,0,2,1,3,1,Gd,9,Typ,2,TA,Attchd,1994.00,RFn,2,712,TA,TA,Y,186,32,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,325000
413,414,30,RM,56.00,8960,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Artery,Norm,1Fam,1Story,5,6,1927,1950,Gable,CompShg,WdShing,Wd Shng,NaN,0.00,TA,TA,CBlock,TA,TA,No,Unf,0,Unf,0,1008,1008,GasA,Gd,Y,FuseA,1028,0,0,1028,0,0,1,0,2,1,TA,5,Typ,1,Gd,Detchd,1927.00,Unf,2,360,TA,TA,Y,0,0,130,0,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal,115000
522,523,50,RM,50.00,5000,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,BrkSide,Feedr,Norm,1Fam,1.5Fin,6,7,1947,1950,Gable,CompShg,CemntBd,CmentBd,NaN,0.00,TA,Gd,CBlock,TA,TA,No,ALQ,399,Unf,0,605,1004,GasA,Ex,Y,SBrkr,1004,660,0,1664,0,0,2,0,3,1,TA,7,Typ,2,Gd,Detchd,1950.00,Unf,2,420,TA,TA,Y,0,24,36,0,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal,159000
1036,1037,20,RL,89.00,12898,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,Timber,Norm,Norm,1Fam,1Story,9,5,2007,2008,Hip,CompShg,VinylSd,VinylSd,Stone,70.00,Gd,TA,PConc,Ex,TA,Gd,GLQ,1022,Unf,0,598,1620,GasA,Ex,Y,SBrkr,1620,0,0,1620,1,0,2,0,2,1,Ex,6,Typ,1,Ex,Attchd,2008.00,Fin,3,912,TA,TA,Y,228,0,0,0,0,0,NaN,NaN,NaN,0,9,2009,WD,Normal,315500



Dataset Information

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 1

## 🧪 Create Working Dataset

The original dataset (`df`) is preserved throughout the notebook.

All preprocessing operations are performed on a separate copy (`df_clean`) to ensure:

- Original dataset remains unchanged.
- Easier debugging.
- Reproducibility.
- Safe experimentation.

In [49]:
# ============================================================
# Create Working Copy
# ============================================================

df_clean = df.copy(deep=True)

print("✓ Working dataset created successfully.\n")

print(f"Original Dataset ID : {id(df)}")
print(f"Working Dataset ID  : {id(df_clean)}")

print("\nDatasets are independent:",
      id(df) != id(df_clean))


✓ Working dataset created successfully.

Original Dataset ID : 1904739676240
Working Dataset ID  : 1904722129408

Datasets are independent: True


# 🔍 Missing Value Analysis

Missing values are analyzed before any treatment is applied.

The analysis includes:

- Total missing values
- Missing percentage
- Missing feature ranking
- Numerical missing values
- Categorical missing values
- Recommended treatment strategy

This helps us understand the nature of missingness before deciding how to handle it.

In [50]:
# ============================================================
# Missing Value Analysis
# ============================================================

missing_counts = df_clean.isnull().sum()

missing_percent = (
    missing_counts / len(df_clean)
) * 100

missing_report = (
    pd.DataFrame({
        "Missing Values": missing_counts,
        "Percentage": missing_percent
    })
    .query("`Missing Values` > 0")
    .sort_values(
        by="Missing Values",
        ascending=False
    )
)

print("=" * 75)
print("MISSING VALUE ANALYSIS REPORT")
print("=" * 75)

print(f"\nTotal Missing Values : {missing_counts.sum()}")
print(f"Columns with Missing Values : {missing_report.shape[0]}")

display(missing_report)

# ------------------------------------------------------------
# Numerical Missing Features
# ------------------------------------------------------------

numeric_missing = (
    df_clean
    .select_dtypes(include=np.number)
    .isnull()
    .sum()
)

numeric_missing = numeric_missing[numeric_missing > 0]

print("\nNumerical Features with Missing Values\n")

display(numeric_missing.to_frame("Missing Values"))

# ------------------------------------------------------------
# Categorical Missing Features
# ------------------------------------------------------------

categorical_missing = (
    df_clean
    .select_dtypes(include="object")
    .isnull()
    .sum()
)

categorical_missing = categorical_missing[categorical_missing > 0]

print("\nCategorical Features with Missing Values\n")

display(categorical_missing.to_frame("Missing Values"))

# ------------------------------------------------------------
# Missing Value Summary
# ------------------------------------------------------------

print("\nSummary")

print(f"Numerical Features with Missing Values   : {len(numeric_missing)}")
print(f"Categorical Features with Missing Values : {len(categorical_missing)}")


MISSING VALUE ANALYSIS REPORT

Total Missing Values : 7829
Columns with Missing Values : 19


,Missing Values,Percentage
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55



Numerical Features with Missing Values



,Missing Values
LotFrontage,259
MasVnrArea,8
GarageYrBlt,81



Categorical Features with Missing Values



,Missing Values
Alley,1369
MasVnrType,872
BsmtQual,37
BsmtCond,37
BsmtExposure,38
BsmtFinType1,37
BsmtFinType2,38
Electrical,1
FireplaceQu,690
GarageType,81



Summary
Numerical Features with Missing Values   : 3
Categorical Features with Missing Values : 16


# 🩺 Missing Value Treatment

This section applies the project's standardized missing value handling pipeline.

The preprocessing strategy is based on the data understanding performed in Notebook 01.

### Strategy

### Categorical Features

Features where missing values represent the absence of a characteristic are filled with `"None"`.

Examples:

- PoolQC
- Alley
- FireplaceQu
- GarageType
- GarageFinish
- GarageQual
- GarageCond
- BsmtQual
- BsmtCond
- BsmtExposure
- BsmtFinType1
- BsmtFinType2
- Fence
- MiscFeature
- MasVnrType

---

### Numerical Features

Different numerical features require different treatment.

| Feature | Strategy |
|----------|----------|
| LotFrontage | Median |
| MasVnrArea | 0 |
| GarageYrBlt | 0 |
| Electrical | Mode |

The preprocessing logic is implemented inside:

`backend/utils/preprocessing_pipeline.py`

In [51]:
# ============================================================
# Missing Value Treatment
# ============================================================

print("=" * 80)
print("MISSING VALUE TREATMENT")
print("=" * 80)

missing_before = df_clean.isnull().sum().sum()

print(f"\nMissing Values Before : {missing_before:,}")

# ------------------------------------------------------------
# Apply Missing Value Pipeline
# ------------------------------------------------------------

df_clean = handle_missing_values(df_clean)

missing_after = df_clean.isnull().sum().sum()

print(f"Missing Values After  : {missing_after:,}")

print("\n✓ Missing value treatment completed successfully.")


MISSING VALUE TREATMENT

Missing Values Before : 7,829
Missing Values After  : 0

✓ Missing value treatment completed successfully.


# ✅ Validation After Missing Value Treatment

After applying the preprocessing pipeline, the dataset is validated to ensure:

- Missing values were handled correctly.
- Dataset dimensions remain unchanged.
- No unexpected NaN values were introduced.
- Data types remain consistent.

In [52]:
# ============================================================
# Validation After Missing Value Treatment
# ============================================================

print("=" * 80)
print("POST-PROCESSING VALIDATION")
print("=" * 80)

remaining_missing = (
    df_clean
    .isnull()
    .sum()
)

remaining_missing = (
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
)

if remaining_missing.empty:
    print("\n🎉 No missing values remain in the dataset.")
else:
    print("\nRemaining Missing Values\n")
    display(remaining_missing.to_frame("Missing Values"))

print("\nDataset Shape :", df_clean.shape)

print("Duplicate Rows :", df_clean.duplicated().sum())

numeric_df = df_clean.select_dtypes(include=np.number)

print(
    "Infinite Values :",
    np.isinf(numeric_df.to_numpy()).sum()
)

print("\nMemory Usage : {:.2f} MB".format(
    df_clean.memory_usage(deep=True).sum() / (1024 ** 2)
))


POST-PROCESSING VALIDATION

🎉 No missing values remain in the dataset.

Dataset Shape : (1460, 81)
Duplicate Rows : 0
Infinite Values : 0

Memory Usage : 3.58 MB


# 🔢 Ordinal Encoding

Certain categorical features have an inherent order (ranking). These features are encoded using ordinal values while preserving their natural ordering.

Examples:

| Original | Encoded |
|----------|---------|
| None | 0 |
| Po | 1 |
| Fa | 2 |
| TA | 3 |
| Gd | 4 |
| Ex | 5 |

The encoding logic is implemented in:

`backend/utils/preprocessing_pipeline.py`

Before encoding, the pipeline validates:

- Required columns exist.
- All category values are supported.
- No unexpected categories are present.

After encoding, the pipeline verifies:

- No new missing values were introduced.
- All encoded columns are numeric.

In [53]:
# ============================================================
# Apply Ordinal Encoding
# ============================================================

print("=" * 80)
print("ORDINAL ENCODING")
print("=" * 80)

missing_before = df_clean.isnull().sum().sum()

print(f"\nMissing Values Before Encoding : {missing_before}")

df_clean = apply_ordinal_encoding(df_clean)

missing_after = df_clean.isnull().sum().sum()

print(f"Missing Values After Encoding  : {missing_after}")

print("\n✓ Ordinal encoding completed successfully.")


ORDINAL ENCODING

Missing Values Before Encoding : 0
INSIDE apply_ordinal_encoding()

Checking category mappings...

✓ All categories successfully validated.

✓ Ordinal encoding completed successfully.

Encoded Columns:
ExterQual            -> int64
ExterCond            -> int64
BsmtQual             -> int64
BsmtCond             -> int64
HeatingQC            -> int64
KitchenQual          -> int64
FireplaceQu          -> int64
GarageQual           -> int64
GarageCond           -> int64
PoolQC               -> int64
BsmtExposure         -> int64
BsmtFinType1         -> int64
BsmtFinType2         -> int64
GarageFinish         -> int64
Functional           -> int64
Missing Values After Encoding  : 0

✓ Ordinal encoding completed successfully.


# ✅ Validation After Ordinal Encoding

The encoded dataset is validated to ensure:

- No missing values were introduced.
- Encoded columns are numeric.
- Dataset dimensions remain unchanged.
- Duplicate rows remain unchanged.
- Infinite values are absent.

In [54]:
# ============================================================
# Validation After Ordinal Encoding
# ============================================================

ordinal_columns = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "GarageFinish",
    "Functional",
]

print("=" * 80)
print("ORDINAL ENCODING VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Remaining Missing Values
# ------------------------------------------------------------

remaining_missing = (
    df_clean[ordinal_columns]
    .isnull()
    .sum()
)

if remaining_missing.sum() == 0:
    print("\n✓ No missing values in ordinal features.")
else:
    print("\nRemaining Missing Values")
    display(
        remaining_missing[
            remaining_missing > 0
        ].to_frame("Missing Values")
    )

# ------------------------------------------------------------
# Data Types
# ------------------------------------------------------------

print("\nEncoded Column Types\n")

display(df_clean[ordinal_columns].dtypes.to_frame("Data Type"))

# ------------------------------------------------------------
# Dataset Validation
# ------------------------------------------------------------

print("\nDataset Shape :", df_clean.shape)

print("Duplicate Rows :", df_clean.duplicated().sum())

numeric_df = df_clean.select_dtypes(include=np.number)

print(
    "Infinite Values :",
    np.isinf(numeric_df.to_numpy()).sum()
)


ORDINAL ENCODING VALIDATION

✓ No missing values in ordinal features.

Encoded Column Types



,Data Type
ExterQual,int64
ExterCond,int64
BsmtQual,int64
BsmtCond,int64
HeatingQC,int64
KitchenQual,int64
FireplaceQu,int64
GarageQual,int64
GarageCond,int64
PoolQC,int64



Dataset Shape : (1460, 81)
Duplicate Rows : 0
Infinite Values : 0


# 📊 Encoded Value Verification

The value distribution of each encoded feature is inspected to confirm that:

- Encoding produced expected numerical values.
- No unexpected labels remain.
- Feature ranges are reasonable.

In [55]:
# ============================================================
# Encoded Value Verification
# ============================================================

print("=" * 80)
print("ENCODED VALUE DISTRIBUTIONS")
print("=" * 80)

for col in ordinal_columns:
    print(f"\n{col}")
    print("-" * len(col))

    print(
        df_clean[col]
        .value_counts(dropna=False)
        .sort_index()
    )


ENCODED VALUE DISTRIBUTIONS

ExterQual
---------
ExterQual
2     14
3    906
4    488
5     52
Name: count, dtype: int64

ExterCond
---------
ExterCond
1       1
2      28
3    1282
4     146
5       3
Name: count, dtype: int64

BsmtQual
--------
BsmtQual
0     37
2     35
3    649
4    618
5    121
Name: count, dtype: int64

BsmtCond
--------
BsmtCond
0      37
1       2
2      45
3    1311
4      65
Name: count, dtype: int64

HeatingQC
---------
HeatingQC
1      1
2     49
3    428
4    241
5    741
Name: count, dtype: int64

KitchenQual
-----------
KitchenQual
2     39
3    735
4    586
5    100
Name: count, dtype: int64

FireplaceQu
-----------
FireplaceQu
0    690
1     20
2     33
3    313
4    380
5     24
Name: count, dtype: int64

GarageQual
----------
GarageQual
0      81
1       3
2      48
3    1311
4      14
5       3
Name: count, dtype: int64

GarageCond
----------
GarageCond
0      81
1       7
2      35
3    1326
4       9
5       2
Name: count, dtype: int64

PoolQC
---

# 🧩 Nominal Encoding

After completing ordinal encoding, the remaining categorical features are nominal, meaning they have **no inherent ordering**.

Examples include:

- MSZoning
- Neighborhood
- Exterior1st
- Exterior2nd
- RoofStyle
- Foundation
- SaleType
- SaleCondition
- HouseStyle
- BldgType

These features are transformed using **One-Hot Encoding**.

The encoding pipeline:

- Automatically detects remaining categorical columns.
- Creates binary indicator features.
- Preserves all categories.
- Validates the final encoded dataset.
- Ensures no missing values or duplicate columns are introduced.

In [56]:
# ============================================================
# Apply Nominal Encoding
# ============================================================

print("=" * 80)
print("NOMINAL ENCODING")
print("=" * 80)

shape_before = df_clean.shape
object_cols_before = df_clean.select_dtypes(include="object").columns.tolist()

print(f"\nObject Columns Before Encoding : {len(object_cols_before)}")
print(f"Dataset Shape Before Encoding  : {shape_before}")

# ------------------------------------------------------------
# Apply Encoding
# ------------------------------------------------------------

df_clean = apply_nominal_encoding(df_clean)

shape_after = df_clean.shape

print(f"\nDataset Shape After Encoding   : {shape_after}")
print(f"New Features Created           : {shape_after[1] - shape_before[1]}")

print("\n✓ Nominal encoding completed successfully.")


NOMINAL ENCODING

Object Columns Before Encoding : 28
Dataset Shape Before Encoding  : (1460, 81)
INSIDE apply_nominal_encoding()

Nominal Features Detected : 28
  • MSZoning
  • Street
  • Alley
  • LotShape
  • LandContour
  • Utilities
  • LotConfig
  • LandSlope
  • Neighborhood
  • Condition1
  • Condition2
  • BldgType
  • HouseStyle
  • RoofStyle
  • RoofMatl
  • Exterior1st
  • Exterior2nd
  • MasVnrType
  • Foundation
  • Heating
  • CentralAir
  • Electrical
  • GarageType
  • PavedDrive
  • Fence
  • MiscFeature
  • SaleType
  • SaleCondition

✓ Nominal encoding completed successfully.

Original Shape : (1460, 81)
Encoded Shape  : (1460, 239)
New Features Added : 158

Dataset Shape After Encoding   : (1460, 239)
New Features Created           : 158

✓ Nominal encoding completed successfully.


# ✅ Validation After Nominal Encoding

The encoded dataset is validated to ensure:

- No categorical (`object`) columns remain.
- No missing values exist.
- Dataset dimensions are consistent.
- Duplicate rows remain unchanged.
- Infinite values are absent.
- All features are suitable for machine learning algorithms.

In [57]:
# ============================================================
# Validation After Nominal Encoding
# ============================================================

print("=" * 80)
print("NOMINAL ENCODING VALIDATION")
print("=" * 80)

remaining_object_columns = (
    df_clean
    .select_dtypes(include="object")
    .columns
    .tolist()
)

if len(remaining_object_columns) == 0:
    print("\n✓ No object-type columns remain.")
else:
    print("\nRemaining Object Columns:")
    print(remaining_object_columns)

print(f"\nDataset Shape : {df_clean.shape}")
print(f"Duplicate Rows : {df_clean.duplicated().sum()}")
print(f"Missing Values : {df_clean.isnull().sum().sum()}")

numeric_df = df_clean.select_dtypes(include=np.number)

print(f"Infinite Values : {np.isinf(numeric_df.to_numpy()).sum()}")

print("\nFeature Type Distribution")

display(
    df_clean.dtypes.value_counts().to_frame("Count")
)


NOMINAL ENCODING VALIDATION

✓ No object-type columns remain.

Dataset Shape : (1460, 239)
Duplicate Rows : 0
Missing Values : 0
Infinite Values : 0

Feature Type Distribution


,Count
int8,186
int64,50
float64,3


# 📏 Feature Scaling

Most machine learning algorithms perform better when continuous numerical features are on a comparable scale.

However, **not every feature should be scaled**.

## Features Excluded from Scaling

The following columns will remain unchanged:

- **Id** (identifier)
- **SalePrice** (target variable)
- **Binary one-hot encoded features (0/1)** generated during nominal encoding

Scaling binary indicator variables is generally unnecessary because they already represent meaningful categorical information.

## Features Selected for Scaling

Only continuous numerical features will be scaled, including examples such as:

- LotFrontage
- LotArea
- MasVnrArea
- GrLivArea
- TotalBsmtSF
- GarageArea
- WoodDeckSF
- OpenPorchSF
- ScreenPorch
- MiscVal
- and other continuous numerical variables.

## Scaling Strategy

To keep the notebook clean and reusable, all scaling logic will be implemented inside:

`backend/utils/preprocessing_pipeline.py`

The notebook will simply call the preprocessing function and validate the results, following the same modular design used for missing value treatment and feature encoding.

## Feature Scaling

This section standardizes the continuous numerical features using the preprocessing pipeline.

Objectives:
- Scale only continuous numerical features.
- Exclude identifier, target, binary, and constant features.
- Preserve the original dataset structure.
- Prepare the dataset for machine learning algorithms.

In [58]:
# ============================================================
# Apply Feature Scaling
# ============================================================

df_clean, scaler, scaled_columns, excluded_columns = (
    scale_numerical_features(
        df_clean,
        scaler="standard",
        exclude_columns=[
            "Id",
            "SalePrice",
            "MSSubClass"
        ],
        verbose=True,
        show_statistics=False
    )
)


INSIDE scale_numerical_features()
✓ Input dataframe validation passed.


FEATURE SCALING SUMMARY
Scaler Selected        : StandardScaler
Numerical Columns      : 239
Continuous Columns     : 50
Binary Columns         : 186
Constant Columns       : 0
Excluded Columns       : 189

Continuous Features Selected:

  • LotFrontage
  • LotArea
  • OverallQual
  • OverallCond
  • YearBuilt
  • YearRemodAdd
  • MasVnrArea
  • ExterQual
  • ExterCond
  • BsmtQual
  • BsmtCond
  • BsmtExposure
  • BsmtFinType1
  • BsmtFinSF1
  • BsmtFinType2
  • BsmtFinSF2
  • BsmtUnfSF
  • TotalBsmtSF
  • HeatingQC
  • 1stFlrSF
  • 2ndFlrSF
  • LowQualFinSF
  • GrLivArea
  • BsmtFullBath
  • BsmtHalfBath
  • FullBath
  • HalfBath
  • BedroomAbvGr
  • KitchenAbvGr
  • KitchenQual
  • TotRmsAbvGrd
  • Functional
  • Fireplaces
  • FireplaceQu
  • GarageYrBlt
  • GarageFinish
  • GarageCars
  • GarageArea
  • GarageQual
  • GarageCond
  • WoodDeckSF
  • OpenPorchSF
  • EnclosedPorch
  • 3SsnPorch
  • ScreenPorch
  • PoolArea
  • PoolQC
  • MiscVal
  • MoSold
  • YrSold

Applying feature

In [59]:
# ============================================================
# Validate Feature Scaling
# ============================================================

print("=" * 60)
print("FEATURE SCALING VALIDATION")
print("=" * 60)

# --------------------------------------------------
# Dataset Information
# --------------------------------------------------

print(f"Dataset Shape          : {df_clean.shape}")
print(f"Total Features         : {df_clean.shape[1]}")
print(f"Scaled Features        : {len(scaled_columns)}")
print(f"Excluded Features      : {len(excluded_columns)}")

# --------------------------------------------------
# Missing & Infinite Values
# --------------------------------------------------

missing_values = df_clean.isnull().sum().sum()

numeric_df = df_clean.select_dtypes(include=[np.number])
infinite_values = np.isinf(numeric_df.to_numpy()).sum()

print(f"\nMissing Values         : {missing_values}")
print(f"Infinite Values        : {infinite_values}")

assert missing_values == 0, "Missing values found after scaling."
assert infinite_values == 0, "Infinite values found after scaling."

# --------------------------------------------------
# Data Type Summary
# --------------------------------------------------

print("\nData Type Distribution")
print("-" * 60)
print(df_clean.dtypes.value_counts())

# --------------------------------------------------
# Validate Scaled Features
# --------------------------------------------------

scaled_means = df_clean[scaled_columns].mean()
scaled_stds = df_clean[scaled_columns].std()

print("\nScaled Feature Statistics")
print("-" * 60)
print(f"Average Mean           : {scaled_means.mean():.6f}")
print(f"Average Std            : {scaled_stds.mean():.6f}")

# --------------------------------------------------
# Column Integrity Check
# --------------------------------------------------

assert len(df_clean.columns) == 239, "Unexpected number of columns."

print("\n✓ Column integrity check passed.")

# --------------------------------------------------
# Final Validation
# --------------------------------------------------

print("\n" + "=" * 60)
print("✓ FEATURE SCALING VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 60)


FEATURE SCALING VALIDATION
Dataset Shape          : (1460, 239)
Total Features         : 239
Scaled Features        : 50
Excluded Features      : 189

Missing Values         : 0
Infinite Values        : 0

Data Type Distribution
------------------------------------------------------------
int8       186
float64     50
int64        3
Name: count, dtype: int64

Scaled Feature Statistics
------------------------------------------------------------
Average Mean           : 0.000000
Average Std            : 1.000343

✓ Column integrity check passed.

✓ FEATURE SCALING VALIDATION COMPLETED SUCCESSFULLY


In [60]:
# ============================================================
# Save Preprocessing Artifacts
# ============================================================

# --------------------------------------------------
# Create Models Directory
# --------------------------------------------------

os.makedirs("../models", exist_ok=True)

# --------------------------------------------------
# Save Standard Scaler
# --------------------------------------------------

joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

# --------------------------------------------------
# Save Scaled Feature List
# --------------------------------------------------

joblib.dump(
    scaled_columns,
    "../models/scaled_columns.pkl"
)

# --------------------------------------------------
# Save Excluded Feature List
# --------------------------------------------------

joblib.dump(
    excluded_columns,
    "../models/excluded_columns.pkl"
)

# --------------------------------------------------
# Save Preprocessing Metadata
# --------------------------------------------------

preprocessing_metadata = {
    "excluded_columns": excluded_columns,
    "scaled_columns": scaled_columns,
    "scaler_name": "StandardScaler",
    "total_features": df_clean.shape[1]
}

joblib.dump(
    preprocessing_metadata,
    MODELS_DIR / "preprocessing_metadata.pkl"
)

# --------------------------------------------------
# Confirmation
# --------------------------------------------------

print("=" * 60)
print("PREPROCESSING ARTIFACTS SAVED")
print("=" * 60)

print("✓ StandardScaler saved")
print("✓ Scaled feature list saved")
print("✓ Excluded feature list saved")
print("✓ Preprocessing metadata saved")

print("\nArtifacts Location")
print("-" * 60)
print("../models/standard_scaler.pkl")
print("../models/scaled_columns.pkl")
print("../models/excluded_columns.pkl")
print("../models/preprocessing_metadata.pkl")


PREPROCESSING ARTIFACTS SAVED
✓ StandardScaler saved
✓ Scaled feature list saved
✓ Excluded feature list saved
✓ Preprocessing metadata saved

Artifacts Location
------------------------------------------------------------
../models/standard_scaler.pkl
../models/scaled_columns.pkl
../models/excluded_columns.pkl
../models/preprocessing_metadata.pkl


In [61]:
# ============================================================
# Final Preprocessing Summary
# ============================================================

print("=" * 70)
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 70)

print(f"Dataset Shape              : {df_clean.shape}")
print(f"Total Features             : {df_clean.shape[1]}")
print(f"Scaled Features            : {len(scaled_columns)}")
print(f"Excluded Features          : {len(excluded_columns)}")

print("\nData Type Distribution")
print("-" * 70)
print(df_clean.dtypes.value_counts())

print("\nSaved Artifacts")
print("-" * 70)
print("✓ standard_scaler.pkl")
print("✓ scaled_columns.pkl")
print("✓ excluded_columns.pkl")
print("✓ preprocessing_metadata.pkl")

print("\nPipeline Status")
print("-" * 70)
print("✓ Missing values handled")
print("✓ Ordinal encoding completed")
print("✓ Nominal encoding completed")
print("✓ Feature scaling completed")
print("✓ Data validation passed")
print("✓ Preprocessing artifacts saved")

print("\n" + "=" * 70)
print("PREPROCESSING PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)


PREPROCESSING PIPELINE SUMMARY
Dataset Shape              : (1460, 239)
Total Features             : 239
Scaled Features            : 50
Excluded Features          : 189

Data Type Distribution
----------------------------------------------------------------------
int8       186
float64     50
int64        3
Name: count, dtype: int64

Saved Artifacts
----------------------------------------------------------------------
✓ standard_scaler.pkl
✓ scaled_columns.pkl
✓ excluded_columns.pkl
✓ preprocessing_metadata.pkl

Pipeline Status
----------------------------------------------------------------------
✓ Missing values handled
✓ Ordinal encoding completed
✓ Nominal encoding completed
✓ Feature scaling completed
✓ Data validation passed
✓ Preprocessing artifacts saved

PREPROCESSING PIPELINE COMPLETED SUCCESSFULLY
